# OOP in Python — From First Principles

> Every concept with runnable code. Built through active recall, not passive reading.
> Run each cell. Predict the output before you run it. That's the learning.

---
## Concept 1 — Why OOP Exists

**Design question:** *How do we organize large programs so that data has a clear owner?*

### Procedural Approach — Data and Functions Are Separate

In [ ]:
# Procedural: anyone can touch the data
balance = 100

def deposit(amount):
    global balance
    balance += amount

def withdraw(amount):
    global balance
    balance -= amount

deposit(50)
withdraw(200)  # no checks — balance goes negative
print(f"Balance: {balance}")  # -50, no one stopped this

### OOP Approach — The Object Owns Its State

In [ ]:
# OOP: the object is the gatekeeper
class BankAccount:
    def __init__(self, balance):
        self._balance = balance

    def deposit(self, amount):
        self._balance += amount

    def withdraw(self, amount):
        if amount > self._balance:
            raise ValueError("Insufficient funds")
        self._balance -= amount

    def get_balance(self):
        return self._balance

acc = BankAccount(100)
acc.deposit(50)
print(f"Balance: {acc.get_balance()}")

# Try withdrawing more than you have:
try:
    acc.withdraw(200)
except ValueError as e:
    print(f"Blocked: {e}")

print(f"Balance after failed withdraw: {acc.get_balance()}")

**Key insight:** `account.withdraw()` expresses responsibility — the account decides whether the operation is allowed. In the procedural version, nothing stopped `balance` from going negative.

---

## Concept 2 — Class and Object

**Design question:** *What is this thing, and how do we create many of them consistently?*

In [ ]:
# Before classes: dictionaries group data, but have no enforcement
dog_dict = {"name": "Buddy", "age": 3, "breed": "Labrador"}

# Typo — no error, silent bug
bad_dog = {"name": "Max", "ag": 5}  # "ag" instead of "age"
print(bad_dog)  # Python doesn't know this is wrong

In [ ]:
# With classes: structure is defined once, enforced always
class Dog:
    def __init__(self, name, age, breed):
        self.name = name
        self.age = age
        self.breed = breed

    def bark(self):
        print(f"{self.name} says Woof!")

dog1 = Dog("Buddy", 3, "Labrador")
dog2 = Dog("Max", 5, "Beagle")

dog1.bark()
dog2.bark()

# type(dog1) returns Dog, not dict — this is foundational
print(f"Type: {type(dog1)}")
print(f"Is dog1 a Dog? {isinstance(dog1, Dog)}")

### Memory Model: Methods Are Shared, Data Is Not

In [ ]:
# Methods live in the class, not in each object
# Both objects share the same bark() method
print(dog1.bark == dog2.bark)  # False — bound to different objects
print(Dog.bark)  # The unbound method lives in the class

# Verify: bark is in the class, not in the object's __dict__
print(f"dog1's own attributes: {dog1.__dict__}")
print(f"Dog class has bark: {'bark' in Dog.__dict__}")

### Variables Are References — Not Objects

In [ ]:
# Two references, one object
a = Dog("Buddy", 3, "Lab")
b = a
b.name = "Max"

# Predict: what does this print?
print(a.name)  # Max — because a and b point to the same object

# Proof: same id
print(f"id(a): {id(a)}")
print(f"id(b): {id(b)}")
print(f"Same object? {a is b}")

---

## Concept 3 — Constructor and `self`

**Design question:** *How is an object actually created, and how does shared behavior know which object it's working on?*

In [ ]:
# __new__ creates the object. __init__ fills it.
class Tracker:
    def __new__(cls):
        print("Step 1: __new__ called — creating empty object")
        instance = super().__new__(cls)
        return instance

    def __init__(self):
        print("Step 2: __init__ called — initializing object")
        self.value = 42

t = Tracker()
print(f"Step 3: value = {t.value}")

### `self` Is Just the Object That Received the Call

In [ ]:
class Dog:
    def __init__(self, name):
        self.name = name

    def introduce(self):
        print(f"I am {self.name}")

buddy = Dog("Buddy")
max_dog = Dog("Max")

# These two calls are identical:
buddy.introduce()       # Python translates to Dog.introduce(buddy)
Dog.introduce(buddy)    # Explicit version — same thing

# self changes every call:
Dog.introduce(max_dog)  # Now self == max_dog

### What Happens Without `self`?

In [ ]:
class Broken:
    def greet():  # no self!
        print("Hello")

b = Broken()
try:
    b.greet()  # Python does Broken.greet(b) — but greet() takes 0 args
except TypeError as e:
    print(f"Error: {e}")

---

## Concept 4 — Encapsulation

**Design question:** *Who is responsible for protecting an object's state?*

In [ ]:
# _ is convention only — Python does nothing
class Employee:
    def __init__(self, salary):
        self._salary = salary  # "please don't touch directly"

emp = Employee(50000)
emp._salary = -100  # Python allows this — convention, not enforcement
print(emp._salary)   # -100

In [ ]:
# __ triggers name mangling
class BankAccount:
    def __init__(self, balance):
        self.__balance = balance  # stored as _BankAccount__balance

acc = BankAccount(100)

# This creates a NEW attribute, doesn't touch the original
acc.__balance = 500

# Two separate attributes now exist:
print(f"acc.__balance = {acc.__balance}")                # 500 (the new one)
print(f"acc._BankAccount__balance = {acc._BankAccount__balance}")  # 100 (original)

# Proof:
print(f"\nAll attributes: {acc.__dict__}")

In [ ]:
# Encapsulation protects invariants, not secrets
class SafeAccount:
    def __init__(self, balance):
        self._balance = balance

    def withdraw(self, amount):
        if amount <= 0:
            raise ValueError("Amount must be positive")
        if amount > self._balance:
            raise ValueError("Insufficient funds")
        self._balance -= amount
        return self._balance

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("Amount must be positive")
        self._balance += amount
        return self._balance

    def get_balance(self):
        return self._balance

acc = SafeAccount(1000)
print(acc.deposit(500))

try:
    acc.withdraw(2000)
except ValueError as e:
    print(f"Blocked: {e}")

print(f"Balance intact: {acc.get_balance()}")

---

## Concept 5 — Static Variables and Methods

**Design question:** *What do we do when data or behavior belongs to the class, not to any one object?*

In [ ]:
# Class variable vs instance variable
class Employee:
    company = "Atlas"  # class variable — one copy, shared
    employee_count = 0

    def __init__(self, name, salary):
        self.name = name      # instance variable — per object
        self.salary = salary
        Employee.employee_count += 1

e1 = Employee("Alice", 50000)
e2 = Employee("Bob", 60000)

print(f"e1.company: {e1.company}")
print(f"e2.company: {e2.company}")
print(f"Total employees: {Employee.employee_count}")

### The Shadowing Trap

In [ ]:
# Classic interview question
class Employee:
    company = "Atlas"

e1 = Employee()
e2 = Employee()

e1.company = "Acme"  # creates a NEW instance variable on e1

print(f"e1.company: {e1.company}")          # Acme (instance var)
print(f"e2.company: {e2.company}")          # Atlas (class var)
print(f"Employee.company: {Employee.company}")  # Atlas (class var)

# Proof: e1 has its own, e2 doesn't
print(f"\ne1.__dict__: {e1.__dict__}")
print(f"e2.__dict__: {e2.__dict__}")

### Three Types of Methods

In [ ]:
class Employee:
    company = "Atlas"

    def __init__(self, name, salary):
        self.name = name
        self.salary = salary

    # Instance method — needs self (one object)
    def display(self):
        print(f"{self.name} at {self.company}, salary: {self.salary}")

    # Class method — needs cls (the class itself)
    @classmethod
    def from_string(cls, data_string):
        name, salary = data_string.split(",")
        return cls(name, int(salary))

    # Static method — needs neither
    @staticmethod
    def is_valid_salary(salary):
        return salary > 0

# Instance method
emp = Employee("Alice", 50000)
emp.display()

# Class method — factory
emp2 = Employee.from_string("Bob,60000")
emp2.display()

# Static method — utility
print(f"Is -100 valid? {Employee.is_valid_salary(-100)}")

In [ ]:
# Why cls(...) matters: subclass safety
class Manager(Employee):
    def __init__(self, name, salary):
        super().__init__(name, salary)
        self.role = "Manager"

# from_string uses cls(...), not Employee(...)
mgr = Manager.from_string("Charlie,80000")
print(f"Type: {type(mgr)}")  # Manager, not Employee!

---

## Concept 6 — Collection of Objects

**Design question:** *How do we manage many objects together, and who owns the group?*

In [ ]:
class Employee:
    def __init__(self, name, salary, department):
        self.name = name
        self.salary = salary
        self.department = department

class Company:
    def __init__(self, name):
        self.name = name
        self.employees = []  # the company owns this collection

    def add_employee(self, emp):
        self.employees.append(emp)

    def total_payroll(self):
        return sum(emp.salary for emp in self.employees)

    def employees_in_department(self, dept):
        return [emp for emp in self.employees if emp.department == dept]

    def highest_paid(self):
        return max(self.employees, key=lambda e: e.salary)

# Build the company
atlas = Company("Atlas Copco")
atlas.add_employee(Employee("Alice", 50000, "HR"))
atlas.add_employee(Employee("Bob", 60000, "IT"))
atlas.add_employee(Employee("Charlie", 70000, "IT"))

print(f"Total payroll: {atlas.total_payroll()}")
print(f"IT employees: {[e.name for e in atlas.employees_in_department('IT')]}")
print(f"Highest paid: {atlas.highest_paid().name}")

In [ ]:
# Collections store references, not copies
emp = atlas.employees[0]
emp.salary = 99999
print(f"Changed via reference: {atlas.employees[0].salary}")  # 99999
print(f"Same object? {emp is atlas.employees[0]}")  # True

---

## Concept 7 — Aggregation and Composition

**Design question:** *When one object contains another, who owns the contained object's lifetime?*

In [ ]:
# Aggregation: contained object has independent lifecycle
class Engine:
    def __init__(self, horsepower):
        self.horsepower = horsepower

    def start(self):
        print(f"Engine ({self.horsepower}hp) started")

class Car:
    def __init__(self, model, engine):
        self.model = model
        self.engine = engine  # passed in — not created here

# Engine exists independently
engine = Engine(300)
car1 = Car("Model X", engine)
car2 = Car("Model Y", engine)  # same engine, two cars

engine.horsepower = 400  # change affects both cars
print(f"car1 engine: {car1.engine.horsepower}hp")
print(f"car2 engine: {car2.engine.horsepower}hp")
print(f"Same engine? {car1.engine is car2.engine}")

In [ ]:
# Composition: contained object's lifecycle is tied to the owner
class Room:
    def __init__(self, name, area):
        self.name = name
        self.area = area

class House:
    def __init__(self, address):
        self.address = address
        # Rooms are created INSIDE the house — composition
        self.rooms = [
            Room("Living Room", 30),
            Room("Kitchen", 15),
            Room("Bedroom", 20),
        ]

    def total_area(self):
        return sum(r.area for r in self.rooms)

house = House("123 Main St")
print(f"Total area: {house.total_area()} sqm")
# If house is deleted, the rooms have no independent purpose

---

## Concept 8 — Inheritance, MRO, and Polymorphism

**Design question:** *What do we do when multiple classes share common behavior? Can different objects respond to the same request differently?*

### Inheritance — Sharing Common Behavior

In [ ]:
class Animal:
    def __init__(self, name):
        self.name = name

    def eat(self):
        print(f"{self.name} is eating")

    def sleep(self):
        print(f"{self.name} is sleeping")

    def speak(self):
        print("...")

class Dog(Animal):
    def speak(self):  # overriding
        print(f"{self.name} says Woof!")

class Cat(Animal):
    def speak(self):  # overriding
        print(f"{self.name} says Meow!")

# Dog and Cat get eat() and sleep() for free
dog = Dog("Buddy")
cat = Cat("Whiskers")

dog.eat()    # inherited from Animal
dog.speak()  # overridden in Dog
cat.speak()  # overridden in Cat

### Methods Are Looked Up Dynamically, Not Copied

In [ ]:
# Prove it: modify Animal.eat() AFTER Dog objects exist
def new_eat(self):
    print(f"{self.name} is eating enthusiastically!")

Animal.eat = new_eat

# Existing dog object uses the NEW method immediately
dog.eat()  # "Buddy is eating enthusiastically!"

# Because Dog looks up eat() in Animal every time — no copying

### `super().__init__()` — Each Class Initializes Its Own State

In [ ]:
class Animal:
    def __init__(self, name):
        self.name = name
        print(f"Animal.__init__: name={name}")

# WRONG: forgetting super()
class BadDog(Animal):
    def __init__(self, name, breed):
        self.breed = breed  # Animal.__init__ never runs!

bad = BadDog("Buddy", "Lab")
try:
    print(bad.name)  # AttributeError!
except AttributeError as e:
    print(f"Error: {e}")

# CORRECT: calling super()
class GoodDog(Animal):
    def __init__(self, name, breed):
        super().__init__(name)  # let Animal do its job
        self.breed = breed

good = GoodDog("Buddy", "Lab")
print(f"name={good.name}, breed={good.breed}")

### Overriding vs Extending

In [ ]:
class Animal:
    def __init__(self, name):
        self.name = name

    def eat(self):
        print(f"{self.name}: Eating food")

# Overriding: completely replaces parent
class Snake(Animal):
    def eat(self):
        print(f"{self.name}: Swallowing prey whole")

# Extending: reuses parent + adds more
class Dog(Animal):
    def eat(self):
        super().eat()  # call parent first
        print(f"{self.name}: Wagging tail while eating")

Snake("Snek").eat()
print("---")
Dog("Buddy").eat()

### The Diamond Problem and MRO

In [ ]:
class A:
    def greet(self):
        print("A")

class B(A):
    def greet(self):
        print("B")

class C(A):
    def greet(self):
        print("C")

class D(B, C):
    pass  # no greet — which parent wins?

d = D()
d.greet()  # Predict before running!

# Python's MRO determines the order:
print(f"\nMRO: {[cls.__name__ for cls in D.mro()]}")

### Polymorphism — Same Call, Different Behavior

In [ ]:
class Animal:
    def __init__(self, name):
        self.name = name

    def speak(self):
        raise NotImplementedError

class Dog(Animal):
    def speak(self):
        return "Woof!"

class Cat(Animal):
    def speak(self):
        return "Meow!"

class Cow(Animal):
    def speak(self):
        return "Moo!"

# Polymorphism in action: same method call, different behavior
animals = [Dog("Buddy"), Cat("Whiskers"), Cow("Bessie")]

for animal in animals:
    print(f"{animal.name} says {animal.speak()}")

# The loop doesn't care what subclass each animal is
# Each object decides how to respond — that's polymorphism

### Operator Overloading — Teaching Python Your Custom Types

In [ ]:
class Vector:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __add__(self, other):
        return Vector(self.x + other.x, self.y + other.y)

    def __eq__(self, other):
        return self.x == other.x and self.y == other.y

    def __str__(self):
        return f"Vector({self.x}, {self.y})"

    def __repr__(self):
        return self.__str__()

v1 = Vector(1, 2)
v2 = Vector(3, 4)

v3 = v1 + v2   # calls v1.__add__(v2)
print(v3)       # calls v3.__str__()

print(f"v1 == v2? {v1 == v2}")           # calls __eq__
print(f"v1 == Vector(1,2)? {v1 == Vector(1, 2)}")

---

## Concept 9 — Pass by Reference

**Design question:** *When you pass an object into a function, can the function change it permanently?*

In [ ]:
# Mutating the object: visible outside
def add_item(lst):
    lst.append("new")

my_list = [1, 2, 3]
add_item(my_list)
print(my_list)  # [1, 2, 3, 'new'] — changed!

# Reassigning the variable: NOT visible outside
def replace_list(lst):
    lst = [100, 200]  # local variable now points elsewhere

my_list2 = [1, 2, 3]
replace_list(my_list2)
print(my_list2)  # [1, 2, 3] — unchanged!

In [ ]:
# Immutable types: reassignment is the only option
def try_increment(n):
    n += 1  # creates a new int, doesn't modify the original
    print(f"Inside function: n = {n}")

x = 5
try_increment(x)
print(f"Outside function: x = {x}")  # still 5

---

## Concept 10 — Abstraction

**Design question:** *How much of an object's internal complexity should the user actually see?*

In [ ]:
# Abstraction: simple interface hiding complex implementation
class CoffeeMachine:
    def make_coffee(self):
        """The only method the user needs to know."""
        self._heat_water()
        self._grind_beans()
        self._brew()
        self._pour()
        print("Coffee ready!")

    def _heat_water(self):
        print("  Heating water to 92°C")

    def _grind_beans(self):
        print("  Grinding beans (medium coarse)")

    def _brew(self):
        print("  Brewing under 9 bars of pressure")

    def _pour(self):
        print("  Pouring into cup")

# User only needs this one line:
machine = CoffeeMachine()
machine.make_coffee()

In [ ]:
# Abstract Base Classes: enforcing a contract
from abc import ABC, abstractmethod

class Shape(ABC):
    @abstractmethod
    def area(self):
        pass

    @abstractmethod
    def perimeter(self):
        pass

class Circle(Shape):
    def __init__(self, radius):
        self.radius = radius

    def area(self):
        return 3.14159 * self.radius ** 2

    def perimeter(self):
        return 2 * 3.14159 * self.radius

class Rectangle(Shape):
    def __init__(self, width, height):
        self.width = width
        self.height = height

    def area(self):
        return self.width * self.height

    def perimeter(self):
        return 2 * (self.width + self.height)

# Can't instantiate abstract class:
try:
    s = Shape()
except TypeError as e:
    print(f"Blocked: {e}")

# Subclasses that fulfill the contract work fine:
shapes = [Circle(5), Rectangle(4, 6)]
for s in shapes:
    print(f"{s.__class__.__name__}: area={s.area():.2f}, perimeter={s.perimeter():.2f}")

---

## Quick Reference

| Concept | Core Idea |
|---|---|
| **Class** | Reusable definition of a type |
| **Object** | One concrete instance with its own state |
| **`self`** | The object receiving the method call |
| **Encapsulation** | Object protects its own state via controlled access |
| **Abstraction** | Simple interface hiding complex implementation |
| **Class Variable** | Shared by all objects, lives in the class |
| **`@staticmethod`** | Needs neither `self` nor `cls` |
| **`@classmethod`** | Receives `cls`, used for factory methods |
| **Aggregation** | "I use this object" — independent lifecycle |
| **Composition** | "I own this object" — tied lifecycle |
| **Inheritance** | Share common behavior via dynamic lookup |
| **`super()`** | Let each class in the chain do its own initialization |
| **MRO** | Deterministic lookup order (C3 linearization) |
| **Polymorphism** | Same call, different behavior, decided at runtime |
| **Operator Overloading** | `__add__`, `__eq__`, `__str__` — teach Python your type |